# FER-YOLO-Mamba - Training Notebook

Runs on **Kaggle GPU** or any CUDA GPU. Training config **auto-scales to VRAM**.

### Quick start on Kaggle
1. **+ Add data** — add the FER-YOLO code repo as a dataset.
2. **Settings → Accelerator → GPU** (T4 x1 is fine).
3. **Settings → Internet → On** (needed to install deps + download dataset).
4. Leave `DATASET_ROOT` / `CODE_ROOT` as `'auto'` and **Run All**.

> **Dataset source:** The RAF-DB dataset is downloaded from Google Drive in Cell 5.
> If you already added the dataset as a Kaggle dataset, it will be detected automatically
> and the download is skipped.

### Quick start locally / on RunPod
Set `DATASET_ROOT` and `CODE_ROOT` to real paths in Cell 1, then run all cells.
Or just use `bash runpod_setup.sh` on RunPod for a one-click experience.


In [ ]:
# ============================================================
# CELL 1 — CONFIGURE PATHS
# ============================================================
import os, shutil, sys

# On a LOCAL / RunPod machine set these to your actual paths, e.g.:
#   CODE_ROOT    = '/workspace/FER-YOLO'
#   DATASET_ROOT = '/workspace'          # parent that contains basic-*/
#                                         # (or 'auto' to trigger GDrive download)
# On KAGGLE: leave both as 'auto'.

CODE_ROOT    = 'auto'
DATASET_ROOT = 'auto'

IS_KAGGLE = os.path.exists('/kaggle')
WORK_DIR  = '/kaggle/working' if IS_KAGGLE else os.path.abspath('.')
print(f'Environment : {"Kaggle" if IS_KAGGLE else "Local / RunPod"}')
print(f'Working dir : {WORK_DIR}')

if CODE_ROOT == 'auto':
    if not IS_KAGGLE:
        raise ValueError('Running locally but CODE_ROOT is still "auto". Set it in Cell 1.')
    _code_root = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if {'nets', 'utils', 'model_data'} <= set(dirs):
            _code_root = root
            break
    assert _code_root, 'Code root (nets/, utils/, model_data/) not found in /kaggle/input'
    CODE_ROOT = _code_root

# DATASET_ROOT: 'auto' means we'll try Kaggle inputs first, then GDrive (Cell 5)
if DATASET_ROOT != 'auto':
    RAF_ROOT = None
    for root, dirs, files in os.walk(DATASET_ROOT):
        if {'EmoLabel', 'Annotation', 'Image'} <= set(dirs):
            RAF_ROOT = root
            break
    assert RAF_ROOT, f'RAF-DB root not found inside {DATASET_ROOT}'
else:
    # Try to find in Kaggle inputs; if not found we download in Cell 5
    RAF_ROOT = None
    if IS_KAGGLE:
        for root, dirs, files in os.walk('/kaggle/input'):
            if {'EmoLabel', 'Annotation', 'Image'} <= set(dirs):
                RAF_ROOT = root
                break
    if RAF_ROOT:
        print(f'RAF-DB found in Kaggle inputs: {RAF_ROOT}')
    else:
        print('RAF-DB not found in Kaggle inputs — will download from Google Drive in Cell 5.')

print(f'Code root   : {CODE_ROOT}')
print(f'RAF-DB root : {RAF_ROOT or "(will download)"}')


In [ ]:
# ============================================================
# CELL 2 — GPU CHECK
# ============================================================
import subprocess
print(subprocess.getoutput('nvidia-smi'))
import torch
print(f'\nPyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU      : {p.name}')
    print(f'VRAM     : {p.total_memory / 1e9:.1f} GB')
else:
    print('GPU      : NOT FOUND — enable GPU accelerator in Settings!')


In [ ]:
# ============================================================
# CELL 3 — COPY CODE TO WRITABLE WORKING DIRECTORY
# ============================================================
_SKIP = {'.git', '__pycache__', '.ipynb_checkpoints', 'logs', 'dataset'}

def _skip(name):
    return name in _SKIP or name.startswith('basic-')

if IS_KAGGLE or os.path.abspath(CODE_ROOT) != os.path.abspath(WORK_DIR):
    for item in os.listdir(CODE_ROOT):
        if _skip(item):
            continue
        src = os.path.join(CODE_ROOT, item)
        dst = os.path.join(WORK_DIR, item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
    print('Code copied to working dir.')
else:
    print('Code already in working dir; skipping copy.')

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print('Working dir :', os.getcwd())
print('Contents    :', sorted(p for p in os.listdir('.') if not p.startswith('.')))


In [ ]:
# ============================================================
# CELL 4 — INSTALL DEPENDENCIES
# Requires Internet = On  (Settings → Internet)
# install_mamba.py downloads PREBUILT wheels matched to this
# torch/CUDA/python — no slow source compile.
# ============================================================
!pip install -q -r requirements.txt
!python install_mamba.py

import importlib
for mod in ['selective_scan_cuda', 'causal_conv1d_cuda']:
    try:
        importlib.import_module(mod)
        print(f'{mod} - OK')
    except Exception as e:
        print(f'{mod} - FAILED: {e}')


In [ ]:
# ============================================================
# CELL 5 — DATASET  (auto-download from Google Drive if needed)
# ============================================================
# If RAF_ROOT was detected in Cell 1 (Kaggle dataset already added),
# this cell just confirms it. Otherwise it downloads from Google Drive.
# Requires Internet = On.
# ============================================================

GDRIVE_URL = (
    'https://drive.google.com/drive/folders/'
    '0B4E10azXECctRUgwVmFPblFUdUE'
    '?resourcekey=0-SCrrCMK2lc4IDmhDsDKhRw'
)

if RAF_ROOT and os.path.isdir(os.path.join(RAF_ROOT, 'EmoLabel')):
    print(f'Using existing dataset: {RAF_ROOT}')
else:
    print('Downloading RAF-DB from Google Drive (~1.8 GB) ...')
    !pip install -q gdown
    import gdown, time
    _dl_dir = os.path.join(WORK_DIR, 'dataset')
    os.makedirs(_dl_dir, exist_ok=True)
    try:
        gdown.download_folder(GDRIVE_URL, output=_dl_dir, quiet=False, use_cookies=False)
    except Exception as e:
        print(f'Download error: {e}')
        print('If quota exceeded, try again in 24h or upload the dataset manually.')
        raise
    # Auto-detect RAF root in downloaded folder
    RAF_ROOT = None
    for root, dirs, files in os.walk(_dl_dir):
        if {'EmoLabel', 'Annotation', 'Image'} <= set(dirs):
            RAF_ROOT = root
            break
    assert RAF_ROOT, (
        f'RAF-DB structure not found under {_dl_dir}. '
        'Check downloaded folder contents and set RAF_ROOT manually.'
    )
    print(f'RAF-DB root detected: {RAF_ROOT}')

print(f'\nRAF-DB root: {RAF_ROOT}')
print(f'EmoLabel   : {os.listdir(os.path.join(RAF_ROOT, "EmoLabel"))}')


In [ ]:
# ============================================================
# CELL 6 — GENERATE ANNOTATION FILES
# ============================================================
from convert_raf_to_yolo import convert

convert(RAF_ROOT, WORK_DIR)

for fname in ['raf_train.txt', 'raf_val.txt']:
    with open(os.path.join(WORK_DIR, fname)) as f:
        lines = f.readlines()
    print(f'{fname}: {len(lines)} samples')
    print(f'  sample: {lines[0].strip()[:100]}')


In [ ]:
# ============================================================
# CELL 7 — CHECK PRETRAINED WEIGHTS
# ============================================================
weights = os.path.join(WORK_DIR, 'model_data', 'yolox_s.pth')
if os.path.exists(weights):
    print(f'Weights : {weights}  ({os.path.getsize(weights)/1e6:.1f} MB)')
else:
    print('NOTE    : yolox_s.pth not found — training from scratch (fine).')

import torch
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'\nVRAM    : {vram:.1f} GB  — training config will be auto-selected')
    if vram >= 40:   print('Profile : 640px  bs=64/32  (40GB+)')
    elif vram >= 22: print('Profile : 640px  bs=32/16  (24GB)')
    elif vram >= 14: print('Profile : 640px  bs=16/8   (16GB T4)')
    else:            print('Profile : 512px  bs=8/4    (<12GB)')
print('\nOverride any setting via %env in Cell 8, e.g.:')
print('  %env EPOCHS=2   %env UNFREEZE_BATCH=4')


In [ ]:
# ============================================================
# CELL 8 — START TRAINING
# ============================================================
# Uncomment to override settings (quick smoke test):
# %env EPOCHS=2
# %env UNFREEZE_BATCH=4
# %env FREEZE_BATCH=8
!python train_kaggle.py


In [ ]:
# ============================================================
# CELL 9 — RESULTS / CHECKPOINTS
# ============================================================
import glob
checkpoints = sorted(glob.glob('logs/**/*.pth', recursive=True))
print(f'Checkpoints saved ({len(checkpoints)} total):')
for ckpt in checkpoints[-5:]:
    print(f'  {ckpt}  ({os.path.getsize(ckpt)/1e6:.1f} MB)')
